# Lab: Making measurements at an ancient Martian lake with Python GIS tools
## Jezero crater, an open-basin paleolake

About 3.5–3.7 billion years ago, rivers on Mars filled impact craters with water. Some of these crater lakes filled until they **overflowed**: the water spilled over the lowest point of the crater rim and cut an **outlet canyon** into it. In this lab you will make measurements at one of the best-studied examples, **Jezero crater**. Jezero is the landing site of NASA's *Perseverance* rover.

*This lab was written as Lab 4 for MIT course 12.621, Remote Sensing: Physical Principles and Techniques.*

You will use real elevation data and the standard Python geospatial stack:

| Package | What we use it for |
|---|---|
| `rioxarray` / `xarray` | reading rasters, CRS information, reprojection, clipping |
| `pandas` | tables (rover chemistry data in Part 5) |
| `osgeo.gdal` / `osgeo.ogr` | generating contour polygons |
| `geopandas` / `shapely` | vector geometries: polygons, lines, areas |
| `pyproj` | coordinate reference systems |
| `scipy`, `numpy`, `matplotlib` | number crunching and plotting |

### Structure
| Part | Topic |
|---|---|
| 0 | Setup |
| 1 | Coordinate reference systems (CRS) |
| 2 | Lake morphometry: extent, area, volume, depth |
| 3 | Inlet and outlet valleys: profiles, slope, eroded volume |
| 4 | Science analysis: how much water drained when the outlet was carved? |
| 5 | Following Perseverance: the rover's traverse and SuperCam rock chemistry |
| 6 | Discussion questions |

### How to work through the lab
* Run the cells **in order** (Shift + Enter).
* Cells marked **✏️ TODO** need you to fill in code. Replace `...` or `# YOUR CODE HERE` with your own code.
* Questions marked **❓** should be answered in the markdown cell below them.
* Some figures are **interactive**: you click on them to draw lines and polygons. This needs the `ipympl` package (see Part 0).
* Part 5 also needs `astropy` (to read a FITS file).

---
# Part 0 — Setup

The next cell turns on **interactive** matplotlib figures (`%matplotlib widget`). It must print `interactive figures: ON`. If it doesn't, install `ipympl` (`pip install ipympl` or `conda install -c conda-forge ipympl`), **restart Jupyter** and run the notebook again.

In [ ]:
import matplotlib
try:
    import ipympl  # noqa: F401  -- provides the interactive "widget" backend
    get_ipython().run_line_magic("matplotlib", "widget")
    print("interactive figures: ON")
except ImportError:
    get_ipython().run_line_magic("matplotlib", "inline")
    print("⚠️  interactive figures: OFF -- ipympl is not installed. Install it, restart Jupyter, "
          "and re-run; otherwise you won't be able to draw on the maps in Parts 2-3.")

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xarray as xr
import rioxarray 
from rasterio.enums import Resampling
import geopandas as gpd
from shapely.geometry import Point
from pyproj import CRS
from osgeo import gdal, ogr, osr
from scipy.interpolate import griddata
from helper_functions import * 

gdal.UseExceptions()
ogr.UseExceptions()
warnings.filterwarnings("ignore", category=RuntimeWarning)  # quieten all-NaN warnings
plt.rcParams.update({"figure.dpi": 90, "font.size": 9})

In [ ]:
# ---- File locations and constants ----
DATA_DIR = Path("jezero_data")             # folder containing the data files
CTX_PATH = DATA_DIR / "jezero_CTX_DEM.tif"
MOLA_PATH = DATA_DIR / "jezero_MOLA_HRSC_DEM.tif"
PREBREACH_PATH = DATA_DIR / "jezero_prebreach.gpkg"
WAYPOINTS_PATH = DATA_DIR / "perseverance_waypoints.json"          # Part 5
TRAVERSE_PATH = DATA_DIR / "perseverance_traverse_jezero.gpkg"     # Part 5
LIBS_PATH = DATA_DIR / "supercam_libs_moc.csv"                    # Part 5
SPECTRUM_PATH = DATA_DIR / "supercam_mingsbight_spectrum.fits"     # Part 5
OUT_DIR = Path("outputs")                  # where we save files we create
OUT_DIR.mkdir(exist_ok=True)

# Approximate centre of Jezero crater (planetocentric latitude, east longitude)
JEZERO_LON, JEZERO_LAT = 77.58, 18.38

---
# Part 1 — Coordinate reference systems (CRS)

Every georeferenced dataset needs a **coordinate reference system** that links its numbers to places on the planet. A CRS bundles together:

1. **A model of the planet's shape** (the *datum*). For Mars we usually use a sphere with radius $R = 3{,}396{,}190$ m (the IAU/"Mars 2000" sphere). Earth data use other models (e.g. WGS84), so you can't mix Earth CRSs (like EPSG codes) with Mars data!
2. **A coordinate system**, which is either
   * **geographic**: longitude and latitude in *degrees* on the sphere, or
   * **projected**: $x, y$ in *metres* on a flat map. A **map projection** converts (longitude, latitude) → $(x, y)$.

No flat map can preserve everything. Each projection distorts some combination of **distance, area, shape and direction**. If we measure areas and volumes on a map, we have to know which of these our projection gets wrong.

### The data
We will use two DEMs (digital elevation models: rasters in which every pixel value is an elevation, in metres, relative to the Mars areoid, the Martian equivalent of "sea level"):

* **CTX DEM, 20 m/pixel**: `jezero_data/jezero_CTX_DEM.tif`, the Mars 2020 *Science Investigation* CTX DEM mosaic of Jezero crater. It was produced using stereophotogrammetry for the Mars 2020 *Perseverance* team from stereo images taken by the Context Camera (CTX) on Mars Reconnaissance Orbiter.
* **MOLA–HRSC blended DEM, 200 m/pixel**: `jezero_data/jezero_MOLA_HRSC_DEM.tif`. This is a regional cut-out of the global DEM that blends Mars Global Surveyor laser altimetry (MOLA) with Mars Express stereo images (HRSC).

## 1.1 Opening a raster with `rioxarray`

`rioxarray.open_rasterio()` loads a raster as an `xarray.DataArray`. This is a NumPy array that carries its **x/y coordinates** and its **CRS** with it, so we can select data by map coordinates instead of pixel indices. All the georeferencing information is available through the `.rio` accessor.

✏️ **TODO 1** Open the CTX DEM (`CTX_PATH`) with [`rioxarray.open_rasterio()`](https://corteva.github.io/rioxarray/stable/rioxarray.html#rioxarray-open-rasterio) and call the result `dem`.
* Pass `masked=True`, so that no-data pixels become `NaN`.

Note that a GeoTIFF can hold several bands, so the result has three dimensions, `(band, y, x)`. Our DEM has a single band; so we add [`.squeeze("band", drop=True)`](https://docs.xarray.dev/en/stable/generated/xarray.DataArray.squeeze.html) to the end of the line to drop that dimension and get a 2-D `(y, x)` array.

In [ ]:
dem = ...
dem = dem.squeeze("band", drop=True) 
dem.name = "elevation"
print(dem)
print("\nresolution:", dem.rio.resolution())
print("bounds:    ", dem.rio.bounds())
print("affine transform:\n", dem.rio.transform())
print("\nCRS (WKT):\n", CRS(dem.rio.crs).to_wkt(pretty=True))

**How to read this:**
* The **affine transform** converts pixel indices (column, row) into map coordinates (x, y):
  $x = a\cdot\text{col} + b\cdot\text{row} + c$, $\;y = d\cdot\text{col} + e\cdot\text{row} + f$.
  Here $a = 20$ m is the pixel width, $e = -20$ m is the pixel height (negative because row numbers increase *downwards* while $y$ increases northwards), and $(c, f)$ is the map coordinate of the **upper-left corner** of the image.
* The **WKT** ("well-known text") is the full description of the CRS. Look for the ellipsoid, the projection method, and its parameters.

❓ **Q1** Answer from the printout above:
1. What are the units of the pixel size?
2. What shape and size of planet does this CRS assume?

*Your answer here.*

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
plot_basemap(ax, dem, step=4)
km_axes(ax)
ax.set_title("CTX DEM of Jezero crater (native CRS)")
plt.show()

## 1.2 A second dataset in a different CRS

Now let's load the regional MOLA–HRSC DEM and plot both DEMs on the same axes, each using its own coordinates.

In [ ]:
mola = rioxarray.open_rasterio(MOLA_PATH, masked=True).squeeze("band", drop=True).astype("float32")
mola.name = "elevation"
print("MOLA CRS:", mola.rio.crs.to_dict())
print("CTX  CRS:", dem.rio.crs.to_dict())
print("Same CRS?", mola.rio.crs == dem.rio.crs)

fig, ax = plt.subplots(figsize=(9, 5))
ax.imshow(hillshade(mola), cmap="gray", extent=da_extent(mola))
ax.imshow(hillshade(dem[::8, ::8]), cmap="copper", extent=da_extent(dem[::8, ::8]), alpha=0.9)
ax.set_xlim(4.30e6, 4.75e6)
ax.set_ylim(1.01e6, 1.17e6)
km_axes(ax)
ax.set_title("Grey: MOLA–HRSC   Copper: CTX  -- each plotted in its own CRS")
plt.show()

The two DEMs **don't line up**: the CTX DEM lands ~200 km west of where Jezero is in the MOLA map. Both use an *equirectangular* projection (`proj: eqc`) on the same sphere, with $x = R\,\lambda\cos\phi_1$ and $y = R\,\phi$. The only difference is the **standard parallel** $\phi_1$ (`lat_ts`), the latitude where the map is true to scale: 0° for MOLA and 18.47° (Jezero's latitude) for CTX. That one parameter shrinks every CTX $x$ coordinate by $\cos 18.47^\circ \approx 0.95$. The coordinates are not wrong; they just mean different things in the two CRSs. To combine datasets, we must put them in the **same** CRS.

## 1.3 Reprojecting a raster

**Reprojecting** a raster means building a new pixel grid in the target CRS. For each new pixel, we find where it falls in the old raster and **resample** a value there:

* `Resampling.nearest` ("Nearest Neighbor") copies the closest old pixel. Use it for categorical data (geologic units, masks).
* `Resampling.bilinear` / `cubic` ("Bilinear" or "Cubic" interpolation) interpolate smoothly. Use them for continuous data like elevation.
* `Resampling.average` averages all old pixels inside each new one. Use it when transforming to much coarser pixels.

✏️ **TODO 2** Reproject the CTX DEM into the MOLA CRS (`mola.rio.crs`) with bilinear resampling, using [`dem.rio.reproject(...)`](https://corteva.github.io/rioxarray/stable/rioxarray.html#rioxarray.raster_array.RasterArray.reproject). The cell then plots both DEMs again.

In [ ]:
dem_eqc = ...
print("original   :", dem.shape, "pixels, resolution", dem.rio.resolution())
print("reprojected:", dem_eqc.shape, "pixels, resolution", dem_eqc.rio.resolution())

fig, ax = plt.subplots(figsize=(6, 5))
ax.imshow(hillshade(mola), cmap="gray", extent=da_extent(mola))
ax.imshow(hillshade(dem_eqc[::8, ::8]), cmap="copper", extent=da_extent(dem_eqc[::8, ::8]), alpha=0.6)
left, right, bottom, top = da_extent(dem_eqc)
ax.set_xlim(left - 20e3, right + 20e3)
ax.set_ylim(bottom - 20e3, top + 20e3)
km_axes(ax)
ax.set_title("After reprojection: CTX (copper) on MOLA (grey)")
plt.show()

**Which CRS should we measure areas and volumes in?** In an equirectangular projection, the true ground area of a pixel changes with latitude, by a factor of $\cos\phi / \cos\phi_1$. In the MOLA CRS ($\phi_1 = 0°$), pixels at Jezero's latitude would be counted ~5% too large, so all our areas and volumes would be ~5% too big. The CTX DEM's own CRS is true to scale at $\phi_1 = 18.47°$, Jezero's latitude. It isn't strictly an equal-area projection, but across the lake (≈18.0–18.8°N) the pixel areas are correct to within ±0.3%, and the small errors north and south of the centre cancel out. So **from here on we measure everything in the CTX DEM's own CRS, using `dem`**.

---
# Part 2 — Lake morphometry

## 2.1 Open-basin lakes on Mars

Many craters on Mars contain evidence of ancient lakes. Valley networks (ancient rivers of Mars) carried flowing water into these lakes, and many have deltas at the interface between valley and lake as sediment was deposited. Crater lakes come in two types:

* **Closed-basin lakes** have inlet valleys but **no outlet**. Water left only by evaporation or infiltration into the groundwater.
* **Open-basin lakes** have inlet valleys *and* an **outlet valley** that leaves the crater. Hence, the lake must have filled up to the lowest point on the rim and overflowed, carving the outlet.

**Jezero** (≈45–49 km across, centred near 18.4°N, 77.6°E) is a classic open-basin lake. Ancient river valleys cut through its rim: inlet valleys that brought water in, and an outlet valley that carried overflowing water away. You will work out which is which in Part 3.

<p align="center"><img src="images/jezero_illustration.jpg" width="480"><br><em>Image caption: Illustration of Jezero Crater billions of years ago when it was filled with water. Image credit: NASA/JPL-Caltech</em></p>

## 2.2 Finding the lake level with GDAL contours

If the lake surface was flat at elevation $L$, then the lake covered the part of the crater that lies **below** $L$ and is connected to the crater floor. GDAL's contouring algorithm can turn a DEM into polygons directly:

* `gdal.ContourGenerateEx(band, layer, options=[...])` writes contours into an OGR vector layer.
* `FIXED_LEVELS=L` asks for a contour at elevation $L$ only.
* `POLYGONIZE=YES` returns **filled polygons** (the regions between contour levels) instead of lines. The region *below* $L$ has the attribute `elev_max = L`.

`polygons_below()` wraps this. It writes a GeoPackage (which you can also open in QGIS) and reads it back as a GeoDataFrame with one row per separate polygon. `show_lake()` keeps only the polygon that contains the crater centre and plots it.

In [ ]:
def polygons_below(dem_path, level, out_path):
    """Polygonize the parts of a DEM that lie below `level` (GDAL contouring).
    Returns a GeoDataFrame with one row per separate polygon."""
    out_path = Path(out_path)
    if out_path.exists():
        out_path.unlink()

    src_ds = gdal.Open(str(dem_path))
    band = src_ds.GetRasterBand(1)
    srs = osr.SpatialReference(wkt=src_ds.GetProjection())

    dst_ds = ogr.GetDriverByName("GPKG").CreateDataSource(str(out_path))
    layer = dst_ds.CreateLayer("contours", srs=srs, geom_type=ogr.wkbMultiPolygon)
    layer.CreateField(ogr.FieldDefn("elev_min", ogr.OFTReal))
    layer.CreateField(ogr.FieldDefn("elev_max", ogr.OFTReal))

    options = [f"FIXED_LEVELS={level}", "POLYGONIZE=YES", "ELEV_FIELD_MIN=0", "ELEV_FIELD_MAX=1"]
    if band.GetNoDataValue() is not None:
        options.append(f"NODATA={band.GetNoDataValue()}")
    gdal.ContourGenerateEx(band, layer, options=options)
    dst_ds = None   # closing the datasets flushes everything to disk
    src_ds = None

    gdf = gpd.read_file(out_path)
    below = gdf[np.isclose(gdf["elev_max"], level)]
    return below.explode(index_parts=False).reset_index(drop=True)


CRATER_CENTRE = Point(4361.75e3, 1089.47e3)   # centre of Jezero (77.58°E, 18.38°N) in the CTX DEM's CRS


def show_lake(level):
    """Contour the DEM at `level`, keep the polygon containing the crater centre, and plot it."""
    polys = polygons_below(CTX_PATH, level, OUT_DIR / "lake_candidate.gpkg")
    lake = polys[polys.contains(CRATER_CENTRE)].reset_index(drop=True)
    fig, ax = plt.subplots(figsize=(6.5, 6))
    plot_basemap(ax, dem, step=8, colorbar=False)
    if len(lake):
        lake.plot(ax=ax, color="royalblue", alpha=0.5, edgecolor="navy", lw=0.5)
    else:
        print("The crater centre is above this level -- there is no lake. Try a higher level.")
    km_axes(ax)
    ax.set_title(f"Lake level {level} m")
    plt.show()
    lake["level"] = level
    return lake

✏️ **TODO 3** Find the lake level yourself. Put an elevation (in metres) into `LAKE_LEVEL` and run the cell: it shows you the lake at that level. Repeat with different numbers until you find the **highest level at which the lake still stays inside the crater**. Each run takes a few seconds. *(Hint: the crater floor is at about −2650 m and the rim is mostly above −2300 m.)*

In [ ]:
LAKE_LEVEL = ...   # elevation in metres -- change it and re-run!
lake = show_lake(LAKE_LEVEL)

❓ **Q2** What lake level did you choose, and why?

*Your answer here.*

You have just created a GeoDataFrame `lake` for the lake polygon you made (at the extent of LAKE_LEVEL). A GeoDataFrame is similar to a pandas DataFrame (basically a spreadsheet), except each row of your DataFrame also saves the geometry of that row. Go ahead and examine the GeoDataFrame below:

In [ ]:
lake

Or you can save `lake` as a .gpkg file to open in QGIS

In [ ]:
lake.to_file(OUT_DIR / "jezero_lake.gpkg")      # save your lake polygon (e.g. to open in QGIS)

## 2.3 Lake area

`lake` is a GeoDataFrame in the CTX DEM's CRS, so the area of its polygon in square metres is (to within ~0.1%) its true area on Mars.

✏️ **TODO 4** Use the [`.area`](https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoSeries.area.html) attribute of `lake` to get the lake area in m². (`.area` returns one value per row; `lake` has a single row, so take the first value with `.iloc[0]`.)

In [ ]:
lake_area = ...
print(f"Lake area: {lake_area / 1e6:,.1f} km²")

## 2.4 Lake volume

The water depth at each pixel is $d = L - z$, where $L$ is the lake level and $z$ the DEM elevation. The lake volume is the sum of the depths multiplied by the pixel area:

$$V = \sum_{\text{pixels in lake}} (L - z_i)\,A_\text{pixel}.$$

✏️ **TODO 5**
1. Clip `dem` to the lake polygon with [`dem.rio.clip([lake_geom], crs=dem.rio.crs, drop=True)`](https://corteva.github.io/rioxarray/stable/rioxarray.html#rioxarray.raster_array.RasterArray.clip).
2. Compute the depth map, $L - z$.
3. Compute the volume in m³: add up the depths of all the pixels with [`.sum()`](https://docs.xarray.dev/en/stable/generated/xarray.DataArray.sum.html) and multiply by the area of one pixel, `px_area` (20 m × 20 m = 400 m²). 1 km³ = 10⁹ m³.

In [ ]:
lake_geom = lake.geometry.iloc[0]               # get the geometry of the lake
px_area = abs(np.prod(dem.rio.resolution()))   # area of one pixel (m²)
lake_dem = ...
depth = ...
depth = depth.where(depth > 0)                  # NaN outside the lake / above water
volume = ...
print(f"Lake volume: {volume / 1e9:,.1f} km³")

## 2.5 Lake depth

✏️ **TODO 6** Find the **maximum** and the **mean** water depth. *Hint:* `depth` is an xarray DataArray, so it has [`.max()`](https://docs.xarray.dev/en/stable/generated/xarray.DataArray.max.html) and [`.mean()`](https://docs.xarray.dev/en/stable/generated/xarray.DataArray.mean.html) methods. Wrap each in `float(...)` to get a plain number.

In [ ]:
max_depth = ...
mean_depth = ...
print(f"max depth {max_depth:.0f} m, mean depth {mean_depth:.0f} m")

imax = depth.argmax(dim=["y", "x"])        # argmax gives you the pixel indices of the deepest point
x_max, y_max = float(depth.x[imax["x"]]), float(depth.y[imax["y"]])
fig, ax = plt.subplots(figsize=(7, 6))
plot_basemap(ax, dem, step=8, colorbar=False, alpha=0.2)
sub = depth[::4, ::4]
im = ax.imshow(sub.values, cmap="Blues", extent=da_extent(sub), vmin=0, vmax=350)
plt.colorbar(im, ax=ax, label="Water depth (m)", shrink=0.8)
ax.plot(x_max, y_max, "r*", ms=14, label="maximum depth")
ax.legend(loc="lower left")
km_axes(ax)
ax.set_title(f"Jezero lake at {LAKE_LEVEL} m")
plt.show()

## 2.6 Interactive profile across the lake

Click on the map to draw a line across the diameter of the lake. The right panel shows the elevation profile, with the lake level as a dashed line.

*(Use `u` to undo a point and `c` to clear. To zoom, use the toolbar's zoom button, but switch it off again before clicking points.)*

In [ ]:
lake_profile = LineDrawer(dem, title="Draw a line across the lake to the outlet",
                          ref_levels={"lake level": LAKE_LEVEL})
plt.show()

---
# Part 3 — Inlet and outlet valleys

Two valleys meet the crater at the points labelled **A** and **B** below. One is an inlet that *brought water in*; the other is the outlet that *carried water out*. Your job is to work out which is which from the topography alone.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
plot_basemap(ax, dem, step=4)
lake.boundary.plot(ax=ax, color="navy", lw=0.8)
for name, (x, y), dx in [("A", (4349.0e3, 1096.6e3), -6e3), ("B", (4388.4e3, 1096.4e3), 6e3)]:
    ax.annotate(name, xy=(x, y), xytext=(x + dx, y + 8e3), fontsize=16, fontweight="bold",
                color="crimson", arrowprops=dict(arrowstyle="->", color="crimson", lw=2))
km_axes(ax)
ax.set_title("Two valleys meeting Jezero crater")
plt.show()

## 3.1 Valley profiles

Draw a line along the floor of each valley. **Start at the crater** (where the arrow points) and click your way along the valley floor, *away* from the crater, for as far as you can follow it (several km to tens of km). The first point you click is circled. Use the toolbar's zoom button if you want to zoom in further (switch it off again before clicking).

Valley floors are narrow, so these tools use a **swath profile**: at each point they take the *lowest* elevation within ±100 m of your line. This way the profile follows the valley floor even if your clicks are slightly off-centre.

In [ ]:
chan_A = LineDrawer(dem, title="Valley A (start at the crater)",
                    extent=(4332e3, 4352e3, 1092.5e3, 1106.5e3), step=1, swath_halfwidth=100,
                    ref_levels={"lake level": LAKE_LEVEL}, color="darkorange",
                    figsize=(14, 6.5), width_ratios=(1.6, 1),
                    outline=lake)   # lake outline in navy
plt.show()

In [ ]:
chan_B = LineDrawer(dem, title="Valley B (start at the crater)",
                    extent=(4386e3, 4418e3, 1093.5e3, 1116.5e3), step=1, swath_halfwidth=100,
                    ref_levels={"lake level": LAKE_LEVEL}, color="purple",
                    figsize=(14, 6.5), width_ratios=(1.6, 1),
                    outline=lake)   # lake outline in navy
plt.show()

✏️ **TODO 7** Plot both valley profiles on the same axes.

Plot each profile with [`ax.plot(x, y, color=..., label=...)`](https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.plot.html), with distance in **km** on the x-axis and elevation on the y-axis. Use `label="Valley A"` and `label="Valley B"` so the legend works.

In [ ]:
distance_A, elevation_A = chan_A.get_profile()   # distance and elevation along valley A
distance_B, elevation_B = chan_B.get_profile()   # ... and along valley B

fig, ax = plt.subplots(figsize=(8, 4))
# plot valley A here
# plot valley B here
ax.axhline(LAKE_LEVEL, color="navy", ls="--", lw=1, label=f"lake level ({LAKE_LEVEL} m)")
ax.set_xlabel("Distance from the crater along the valley (km)")
ax.set_ylabel("Elevation (m)")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

❓ **Q3** Which valley is the **outlet** (water flowing *away* from the lake), and which is the **inlet**? Explain using the profiles.

*Your answer here.*

## 3.2 Slope of the outlet

Now you know which valley is the outlet. Measure its average slope by **fitting a straight line** to its profile, $z = m\,d + c$, where $m$ is the gradient (metres of drop per metre of distance; negative when the valley goes downhill). The slope angle is $\theta = \arctan |m|$.

✏️ **TODO 8**
1. Fit the line with [`np.polyfit(distance, elevation, 1)`](https://numpy.org/doc/stable/reference/generated/numpy.polyfit.html), which returns `[m, c]`. 
2. Convert the gradient to a slope angle in **degrees** with [`np.arctan`](https://numpy.org/doc/stable/reference/generated/numpy.arctan.html) and [`np.degrees`](https://numpy.org/doc/stable/reference/generated/numpy.degrees.html).

The cell then plots your fitted line on top of the profile.

In [ ]:
def plot_fit(distance, elevation, m, c, title=""):
    """Plot a profile (d, z in m) and the fitted line z = m*d + c."""
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(distance / 1e3, elevation, color="purple", lw=1, label="outlet profile")
    ax.plot(distance / 1e3, m * distance + c, "k--", lw=1.5, label=f"fit: gradient {m:.4f}")
    ax.set_xlabel("Distance from the crater along the valley (km)")
    ax.set_ylabel("Elevation (m)")
    ax.set_title(title)
    ax.legend()
    ax.grid(alpha=0.3)
    plt.show()


distance_outlet, elevation_outlet = ...   # which profile is the outlet?
ok = ~np.isnan(elevation_outlet)          # filter out any no-data samples
distance_outlet, elevation_outlet = distance_outlet[ok], elevation_outlet[ok]
m, c = ...
outlet_slope_deg = ...
plot_fit(distance_outlet, elevation_outlet, m, c, title=f"Outlet slope: {outlet_slope_deg:.2f}°")

## 3.3 Volume eroded by the outlet canyon

To measure how much rock the outlet removed, we need to know the land surface **before** the canyon was cut. A common approximation:

1. Outline the canyon with a polygon whose edges run along its **banks** (the top edges of the canyon walls).
2. Sample the DEM along that outline. These elevations are assumed to be the pre-incision surface.
3. **Interpolate** across the inside of the polygon to rebuild the "lid" over the canyon.
4. Eroded depth = reconstructed surface − present-day DEM; eroded volume = $\sum$ depth × pixel area.

**Draw the canyon outline** below. Click points along one bank of the outlet canyon, from the rim breach to where the canyon opens onto the plains. Then come back along the other bank, and click on your first point to close the polygon. Afterwards you can drag vertices to adjust them.

In [ ]:
canyon_editor = PolygonEditor(dem, title="Outline the outlet canyon along its banks",
                              extent=(4384e3, 4408e3, 1093.5e3, 1117.5e3), step=1)
plt.show()

In [ ]:
def pre_incision_surface(da, polygon):
    """Rebuild the land surface over a valley: sample `da` along the polygon edge and
    interpolate linearly across the interior. Returns a DataArray clipped to the polygon."""
    res = abs(da.rio.resolution()[0])
    sub = da.rio.clip([polygon], crs=da.rio.crs, drop=True)
    edge = np.array(polygon.exterior.segmentize(res).coords)
    z_edge = da.sel(x=xr.DataArray(edge[:, 0]), y=xr.DataArray(edge[:, 1]), method="nearest").values
    ok = ~np.isnan(z_edge)
    X, Y = np.meshgrid(sub.x.values, sub.y.values)
    surface = griddata(edge[ok], z_edge[ok], (X, Y), method="linear")
    return sub.copy(data=surface).where(sub.notnull())


canyon_poly = canyon_editor.polygon
gpd.GeoSeries([canyon_poly], crs=dem.rio.crs).to_file(OUT_DIR / "canyon_outline.gpkg")   # keep a copy of your outline
canyon_dem = dem.rio.clip([canyon_poly], crs=dem.rio.crs, drop=True)
surface = pre_incision_surface(dem, canyon_poly)

The cell above made two DataArrays on the same grid, covering only the inside of your polygon:
* **`canyon_dem`** is the **present-day** elevation inside the canyon outline (the DEM clipped to your polygon).
* **`surface`** is the **reconstructed pre-erosion** elevation: the elevations along your outline, interpolated across the banks. It is the "original topography" before the canyon was carved.

✏️ **TODO 9** Compute:
1. `eroded`: the thickness of rock removed at each pixel, `surface - canyon_dem`. Set any negative values to zero with [`.clip(min=0)`](https://docs.xarray.dev/en/stable/generated/xarray.DataArray.clip.html).
2. `canyon_volume`: the eroded volume in m³. Use [`.sum()`](https://docs.xarray.dev/en/stable/generated/xarray.DataArray.sum.html) and `px_area`, as in TODO 5.
3. `canyon_max_depth`: the maximum eroded depth in m, with `.max()`.

In [ ]:
# Define eroded (DataArray, m), canyon_volume (m³) and canyon_max_depth (m)
eroded = ...
canyon_volume = ...
canyon_max_depth = ...

print(f"canyon outline area {canyon_poly.area/1e6:.1f} km²")
print(f"eroded volume       {canyon_volume/1e9:.2f} km³")
print(f"max eroded depth    {canyon_max_depth:.0f} m")

fig, ax = plt.subplots(figsize=(7, 6))
plot_basemap(ax, dem, step=1, extent=(4384e3, 4408e3, 1093.5e3, 1117.5e3), colorbar=False, alpha=0.15)
im = ax.imshow(eroded.values, cmap="magma_r", extent=da_extent(eroded))
plt.colorbar(im, ax=ax, label="Eroded depth (m)", shrink=0.8)
ax.plot(*canyon_poly.exterior.xy, color="cyan", lw=1)
km_axes(ax)
ax.set_title("Material removed by the outlet canyon")
plt.show()

❓ **Q4** What are the main sources of uncertainty in this volume? Try moving your outline a little outward or inward and re-running the last two cells. How sensitive is the answer?

*Your answer here.*

---
# Part 4 — Science analysis: how much water drained when the outlet was carved?

The outlet floor at the rim is at ≈ −2400 m today, but that is the level **after** the canyon was cut. When the lake first overflowed, the rim was intact, so the water stood **higher**. Once water started pouring over the rim it eroded a breach. As the breach deepened, the lake drained down to the new, lower outlet. This is an **overflow flood**, like a natural dam failure. The **flood volume** is the water released between the pre-breach and post-breach lake levels:

$$V_\text{flood} = V_\text{pre-breach} - V_\text{post-breach}.$$

<p align="center"><img src="images/flood_volume_schematic.png" width="480"><br><em>Image caption: Schematic of flood volume for an open-basin lake. Image credit: Ze-Wen Koh</em></p>

Goudge et al. (2019) measured this for 24 Martian paleolakes. They found that the volume eroded from the outlet canyon is strongly related to the flood volume, which suggests outlet canyons were carved rapidly, during the floods themselves. Here you'll see whether Jezero fits their trend.

The pre-breach lake is provided as a GeoPackage (`jezero_data/jezero_prebreach.gpkg`). Its attribute `BreachZ_m` is the elevation of the rim before the canyon was eroded ("original outlet level" above, at the breach point). Your **post-breach** lake is the one you measured in Part 2.

✏️ **TODO 10** Load the pre-breach lake.
1. Read the file `PREBREACH_PATH` with [`gpd.read_file()`](https://geopandas.org/en/stable/docs/reference/api/geopandas.read_file.html) and call it `prebreach`.
2. Pull out the pre-breach lake level from the `BreachZ_m` column. Columns work as in pandas: `prebreach["column_name"]` gives the column, and `.iloc[0]` gives its first value (the file has a single row). Wrap it in `float(...)`.
3. The polygon is in the MOLA CRS. Convert it to the CTX DEM's CRS, `dem.rio.crs`, with [`.to_crs()`](https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoDataFrame.to_crs.html).

In [ ]:
prebreach = ...
print("pre-breach polygon CRS:", prebreach.crs.name)
PREBREACH_LEVEL = ...
prebreach = ...   # convert to the CTX DEM's CRS
prebreach_geom = prebreach.geometry.iloc[0]
print(f"pre-breach lake level: {PREBREACH_LEVEL:.0f} m")

Plot the extent of the lake before and after the breach occurred below:

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
plot_basemap(ax, dem, step=8, colorbar=False)
prebreach.boundary.plot(ax=ax, color="crimson", lw=1.2, label=f"pre-breach ({PREBREACH_LEVEL:.0f} m)")
lake.boundary.plot(ax=ax, color="navy", lw=1.2, label=f"post-breach ({LAKE_LEVEL} m)")
ax.legend(loc="lower left")
km_axes(ax)
ax.set_title("Jezero lake before and after the outlet was carved")
plt.show()

✏️ **TODO 11**
1. Compute the **pre-breach volume** `V_pre` in m³, using the same steps as TODO 5 but with `prebreach_geom` and `PREBREACH_LEVEL`.
2. Compute the **flood volume**: `V_pre` minus your post-breach `volume` from TODO 5.

In [ ]:
pre_dem = ...
pre_depth = ...
V_pre = ...
flood_volume = ...

print(f"pre-breach volume  {V_pre/1e9:6.0f} km³")
print(f"post-breach volume {volume/1e9:6.0f} km³")
print(f"flood volume       {flood_volume/1e9:6.0f} km³")

## Does Jezero's canyon fit the trend for other Martian lakes?

Goudge et al. (2019) plotted outlet canyon volume against flood volume for their 24 breached paleolakes (figure below) and fitted a power law:

$$V_\text{canyon} = 0.93\,V_\text{flood}^{\,0.92}, \qquad \text{both volumes in m}^3.$$

<p align="center"><img src="images/goudge2019_fig2c.png" width="380"><br><em>Image caption: Relationship of outlet canyon volume to drained/flood volume of the open-basin paleolake. Image credit: Goudge et al. (2019; Fig. 2C)</em></p>

✏️ **TODO 12** Use the fit to predict the canyon volume, `predicted_canyon_volume`, from your `flood_volume`. Keep everything in **m³**: the fit only works in the units it was made in. (In Python, `x**0.92` raises `x` to the power 0.92.)

The cell then plots your measured canyon volume against the fitted trend.

In [ ]:
predicted_canyon_volume = ...

v = np.logspace(8, 15, 50)
fig, ax = plt.subplots(figsize=(5.5, 4.5))
ax.loglog(v, 0.93 * v**0.92, "k-", lw=1, label="Goudge et al. (2019) fit")
ax.loglog(flood_volume, canyon_volume, "r*", ms=14, label="Jezero, your measurement")
ax.set_xlabel(r"Flood (drained) volume [m$^3$]")
ax.set_ylabel(r"Outlet canyon volume [m$^3$]")
ax.legend(fontsize=8)
ax.grid(alpha=0.3, which="major")
plt.show()

❓ **Q5** How far is your measured canyon volume from the trend from Goudge et al. (2019)? What are possible reasons for how it compares to the trend?

*Your answer here.*

---
# Part 5 — Following Perseverance

*Perseverance* landed on the floor of Jezero on 18 February 2021 (sol 0; a **sol** is a Martian day of 24 h 40 min). Since then it has driven across the crater floor, up the front of the western delta, and up the crater rim. After every drive NASA publishes the rover's position and elevation. In this part you map where the rover has been, see how far it has driven, and look at what its laser found in the rocks along the way.

The rover positions are given in the CTX DEM's own CRS, so we can plot them directly on `dem`.

In [ ]:
wp = gpd.read_file(WAYPOINTS_PATH)
wp = wp[wp["final"] == "y"].sort_values(["sol", "site", "drive"]).reset_index(drop=True)  # end-of-drive positions only
print(f"{len(wp)} end-of-drive positions, up to sol {wp.sol.max()}")

## 5.1 Mapping the traverse

The map shows the rover's track (white line) through its end-of-drive positions, coloured by sol, on a shaded relief of the CTX DEM. The star marks the landing site.

In [ ]:
# map extent
pad = 2500   # buffer of map boundary (in m) around the traverse
ROVER_EXTENT = (wp.easting.min() - pad, wp.easting.max() + pad, wp.northing.min() - pad, wp.northing.max() + pad)

fig, ax = plt.subplots(figsize=(11, 5))
plot_basemap(ax, dem, step=1, extent=ROVER_EXTENT, alpha=0, colorbar=False)   # shaded relief only
ax.plot(wp.easting, wp.northing, color="white", lw=1)
sc = ax.scatter(wp.easting, wp.northing, c=wp.sol, cmap="plasma", s=8, zorder=2)
plt.colorbar(sc, ax=ax, label="sol", shrink=0.8)
ax.plot(wp.easting.iloc[0], wp.northing.iloc[0], "*", ms=14, mfc="yellow", mec="k", label="landing site")
ax.legend(loc="lower right")
km_axes(ax)
ax.set_title("Perseverance traverse on the CTX DEM of Jezero")
plt.show()

## 5.2 Has Perseverance run a marathon?

`jezero_data/perseverance_traverse_jezero.gpkg` contains the rover's path in the CTX DEM's CRS. Here we read it as a geopandas GeoDataFrame, where each row is each drive of the rover. The CTX DEM's CRS is true to scale at Jezero's latitude, so lengths measured in it are real distances on the ground.

In [ ]:
traverse = gpd.read_file(TRAVERSE_PATH)
print(traverse.crs.name, "|", len(traverse), "drives")
print(traverse.head(5)) # prints the first 5 rows of the traverse GeoDataFrame

✏️ **TODO 13**
1. Find the total distance Perseverance has driven, `total_km`, in km. Use the [`.length`](https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoSeries.length.html) attribute of `traverse`, which gives the length of each drive in **metres**, and add them all up with `.sum()`.
2. Look up the length of a marathon, set `MARATHON_KM`, and work out how many marathons the rover has driven.

In [ ]:
MARATHON_KM = ...   # length of a marathon in km
total_km = ...
n_marathons = ...
print(f"Perseverance has driven {total_km:.1f} km = {n_marathons:.2f} marathons, in {wp.sol.max()} sols")

❓ **Q6** Has Perseverance run a marathon? A marathon runner finishes in about 3 hours. How long has Perseverance taken, in Earth years (1 sol = 1.0275 Earth days)? Why do you think it drives so slowly?

*Your answer here.*

## 5.3 Zapping rocks with SuperCam

Perseverance carries its own remote-sensing instrument on its mast. **SuperCam** fires a laser at a rock up to about 7 m away. The laser turns a spot of rock less than 1 mm across into a glowing spark of hot gas (a plasma), and a spectrometer records the light. This is an **emission-line spectrum**, just like the gas tubes in Lab 1: each element gives its own set of lines, so the team can work out what the rock is made of. The file `jezero_data/supercam_libs_moc.csv` gives the result for each laser spot as the amounts of eight oxides, in percent by weight (`SiO2`, `TiO2`, `Al2O3`, `FeOT`, `MgO`, `CaO`, `Na2O`, `K2O`). Characters 5 to 8 of the file name in `cdr_fname` are the sol. Targets whose names start with `scct` are calibration targets carried on the rover, not Martian rocks.

Below is one SuperCam spectrum, of a rock the team named *Mings Bight* on sol 1494, with some of the brightest lines labelled. SuperCam has three spectrometers, one for each panel, and each panel is scaled so that its brightest line is 1.

In [ ]:
from astropy.io import fits

with fits.open(SPECTRUM_PATH) as f:
    wl = f["WAVELENGTH"].data["Wavelength"].astype(float)   # nm (vacuum)
    flux = f["STATISTICS"].data["Mean"].astype(float)        # mean of the laser shots
lines = {"Mg": [279.6, 285.2], "Si": [288.2], "Ca": [393.4, 396.8], "Al": [394.5, 396.3],
         "Na": [589.2], "K": [766.7, 770.1], "O": [777.5]}
ranges = [(240, 342, "UV"), (379, 465, "violet"), (535, 855, "visible and near infrared")]

fig, axes = plt.subplots(1, 3, figsize=(12, 3.8), gridspec_kw={"width_ratios": [102, 86, 320]})
for ax, (lo, hi, name) in zip(axes, ranges):
    m = (wl >= lo) & (wl <= hi)
    ax.plot(wl[m], flux[m] / flux[m].max(), color="k", lw=0.6)
    for el, lams in lines.items():
        for lam in lams:
            if lo < lam < hi:
                i = np.argmax(np.where(abs(wl - lam) < 0.5, flux, -np.inf))
                ax.annotate(el, (wl[i], flux[i] / flux[m].max()), xytext=(0, 3), textcoords="offset points",
                            ha="center", fontsize=8, color="crimson")
    ax.set_xlim(lo, hi)
    ax.set_ylim(-0.02, 1.12)
    ax.set_title(name, fontsize=9)
    ax.set_xlabel("Wavelength (nm)")
axes[0].set_ylabel("Brightness (scaled to 1 in each range)")
fig.suptitle("SuperCam LIBS spectrum of Mings Bight, sol 1494")
plt.tight_layout()
plt.show()

**Why MgO?** Rocks are made of minerals, and the amount of magnesium oxide (MgO) is a good tracer of one mineral in particular, **olivine**, a green magnesium-iron silicate, (Mg,Fe)₂SiO₄. Olivine is one of the first minerals to crystallise when molten rock cools, so it can pile up into olivine-rich layers with 25 to 40 % MgO. Most other volcanic rocks, such as basalt, are made mostly of feldspar and pyroxene and have less than about 10 % MgO. Olivine is also quickly altered by water into clays and carbonates, so it is a key mineral for studying the lake. Like NDVI in Lab 3, which tells you how much vegetation is in a pixel, the MgO value tells you roughly how much olivine is in a laser spot.

✏️ **TODO 14** Compute `mgo_per_sol`, the median `MgO` of all laser spots on each sol, as a pandas Series indexed by sol. *(Hint: `libs.groupby("sol")[...].median()`.)* The cell then puts it on the map of Jezero, on top of the rover's track, with the map coloured by elevation.

In [ ]:
libs = pd.read_csv(LIBS_PATH, skiprows=8)   # the first 8 rows describe the calibration
libs["sol"] = libs.cdr_fname.str[5:9].astype(int)
libs = libs[~libs.cdr_fname.str[39:].str.startswith("scct")]   # drop the rover's calibration targets
print(f"{len(libs)} laser spots, sols {libs.sol.min()} to {libs.sol.max()}")

mgo_per_sol = ...

# rover position on each sol with SuperCam data
spots = pd.merge_asof(mgo_per_sol.rename("MgO").reset_index(),
                      wp[["sol", "easting", "northing", "elev_geoid"]].astype({"sol": "int64"}), on="sol")

fig, ax = plt.subplots(figsize=(12, 5.5))
plot_basemap(ax, dem, step=1, extent=ROVER_EXTENT, alpha=0.6)          # coloured by elevation
ax.plot(wp.easting, wp.northing, color="white", lw=1, zorder=1)
sc = ax.scatter(spots.easting, spots.northing, c=spots.MgO, cmap="YlOrRd", s=22, ec="k", lw=0.5, zorder=2)
ax.plot(wp.easting.iloc[0], wp.northing.iloc[0], "*", ms=14, mfc="yellow", mec="k", label="landing site")
ax.legend(loc="lower right", fontsize=8)
plt.colorbar(sc, ax=ax, label="MgO (%)", shrink=0.8)
km_axes(ax)
ax.set_title("SuperCam MgO along the rover track, on the CTX DEM")
plt.show()

❓ **Q7** Is the floor of Jezero made of one kind of rock, or several? Look at where the amount of MgO changes along the track. Do the changes happen gradually or in steps, and do they line up with changes in elevation (the map colours)? Why might rocks at different heights be different?

*Your answer here.*

---
# Part 6 — Discussion questions

Answer in the markdown cells below each question. Use numbers from your analysis where you can.

**D1.** CTX captures reflected sunlight (passive optical). If you were instead analyzing Jezero crater using a Synthetic Aperture Radar (SAR) system (like Sentinel-1 or Magellan) at a 5 cm wavelength ($\lambda$):
* How would the smooth, fine-grained mudstones of the ancient lakebed appear relative to the blocky, boulder-strewn rim of the crater in a radar image?
* What physical surface property dictates brightness in SAR imagery that is largely invisible in standard CTX optical imagery?

*Your answer here.*

**D2.** Fresh impact craters of this size (~45 km wide) on Mars are **complex craters**: unlike small, bowl-shaped simple craters (depth ≈ 1/5 of the diameter), they have flatter floors, central peaks and terraced walls, and are much shallower relative to their width. A fresh Martian complex crater of diameter $D$ is typically $d \approx 0.36\,D^{0.49}$ deep from rim crest to floor, **with both $d$ and $D$ in kilometres** (Garvin et al., 2003), i.e. $0.36 \times 45^{0.49} \approx 2.3$ km for a crater the size of Jezero. Look at your cross-section profile across the crater (2.6). How deep is Jezero today, from rim crest to floor? Is the floor bowl-shaped or flat? Based on your measurements, roughly how many kilometres of material must be sitting on top of the original crater floor?

*Your answer here.*

---
## References

* Fergason, R. L., Hare, T. M., Mayer, D. P., Galuszka, D. M., Redding, B. L., Smith, E. D., Shinaman, J. R., Cheng, Y., Otero, R. E., 2020, Mars 2020 Terrain Relative Navigation Flight Product Generation: Digital Terrain Model and Orthorectified Image Mosaics, 51st Lunar and Planetary Science Conference, URL: https://www.hou.usra.edu/meetings/lpsc2020/pdf/2020.pdf.
* Fergason, R. L, Hare, T. M., & Laura, J. (2018). HRSC and MOLA Blended Digital Elevation Model at 200m v2. Astrogeology PDS Annex, U.S. Geological Survey. https://astrogeology.usgs.gov/search/map/mars_mgs_mola_mex_hrsc_blended_dem_global_200m
* Goudge, T. A., Fassett, C. I., & Mohrig, D. (2019). Incision of paleolake outlet canyons on Mars from overflow flooding. *Geology*, 47(1), 7–10. https://doi.org/10.1130/G45397.1
* Garvin, J.B., Sakimoto, S.E.H., Frawley, J.J., 2003. Craters on Mars: Global geometric
properties from gridded MOLA topography. In: Sixth International Conference
on Mars. Abstract no. #3277.